# nba_api exploration (task 1.2)

Goal: get familiar with `nba_api`'s endpoints before building the data
pipeline (`fetch.py`, task 1.3), and document what actually works, what
doesn't, and why — rate limiting, timeouts, blocking — so the pipeline is
designed around real constraints instead of assumptions.


In [1]:
import time

import pandas as pd
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import commonplayerinfo, playercareerstats, leaguegamelog


## 1. Static data (bundled with the library, no network call)

`nba_api.stats.static` ships a hardcoded snapshot of all players/teams and
their IDs. This is instant and reliable — useful for resolving a player's
name to the `player_id` that every live endpoint needs.


In [2]:
lebron = players.find_players_by_full_name("LeBron James")
lakers = teams.find_teams_by_full_name("Lakers")
print(lebron)
print(lakers)


[{'id': 2544, 'full_name': 'LeBron James', 'first_name': 'LeBron', 'last_name': 'James', 'is_active': True}]
[{'id': 1610612747, 'full_name': 'Los Angeles Lakers', 'abbreviation': 'LAL', 'nickname': 'Lakers', 'city': 'Los Angeles', 'state': 'California', 'year_founded': 1948}]


## 2. Live endpoints (`stats.nba.com`)

The endpoints the project actually needs data from:

- `CommonPlayerInfo` — bio/profile data for the player profile page (1.6)
- `PlayerCareerStats` — season-by-season stats, needed for both the
  profile page and the comparison page (1.6/1.7)
- `LeagueGameLog` — per-game logs, a candidate source for more granular
  analysis in V2

These all call `stats.nba.com` directly (not a public/documented REST
API — `nba_api` reverse-engineers the endpoints NBA.com's own site uses).
Let's try one and measure what happens.


In [3]:
def try_endpoint(name, fn, timeout=15):
    start = time.time()
    try:
        result = fn(timeout=timeout)
        df = result.get_data_frames()[0]
        elapsed = time.time() - start
        print(f"{name}: OK in {elapsed:.1f}s, {len(df)} rows")
        return df
    except Exception as e:
        elapsed = time.time() - start
        print(f"{name}: FAILED after {elapsed:.1f}s -> {type(e).__name__}: {e}")
        return None


player_id = lebron[0]["id"]  # 2544
df_info = try_endpoint(
    "CommonPlayerInfo",
    lambda timeout: commonplayerinfo.CommonPlayerInfo(player_id=player_id, timeout=timeout),
)


CommonPlayerInfo: FAILED after 18.4s -> ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)


In [4]:
# nba_api's own docs recommend a short delay between calls to avoid
# tripping rate limits, even when a single call succeeds.
time.sleep(0.6)

df_career = try_endpoint(
    "PlayerCareerStats",
    lambda timeout: playercareerstats.PlayerCareerStats(player_id=player_id, timeout=timeout),
)

time.sleep(0.6)

df_gamelog = try_endpoint(
    "LeagueGameLog",
    lambda timeout: leaguegamelog.LeagueGameLog(season="2023-24", timeout=timeout),
)


PlayerCareerStats: FAILED after 15.4s -> ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)


LeagueGameLog: FAILED after 15.4s -> ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=15)


## Findings: rate limiting & blocking

All three live calls above timed out (`ReadTimeout`) rather than
returning data or an explicit error status. This matches a well-known,
documented limitation of `nba_api`:

- `stats.nba.com` sits behind bot-protection that identifies and
  silently drops/stalls requests from datacenter and cloud IP ranges
  (AWS, GCP, Azure, and sandboxed environments like this one all fall
  in that bucket). It does **not** send back a clean `403` — the
  connection just hangs until the client's timeout fires, which is
  exactly what was observed.
- This is an IP-reputation problem, not a code problem: sending the
  right headers (`nba_api` already does this — `Referer`,
  `x-nba-stats-origin`, a browser `User-Agent`) does not help once the
  source IP itself is flagged.
- From a residential/home IP (e.g. running `fetch.py` on a personal
  laptop, which is how this project will actually be used), these same
  calls are expected to succeed — this is a widely reported pattern in
  the `nba_api` community, not specific to this network.

### Design implications for `fetch.py` (task 1.3)

1. **Throttle every live call** with a small delay (`time.sleep`,
   ~0.6-1s) between requests, even on success — the rate limit is real
   and independent of the blocking issue above.
2. **Fail loud, not silent**: wrap each call with a bounded timeout and
   surface a clear error (not a hang) when a request stalls, so a
   pipeline run fails fast instead of hanging for minutes per endpoint.
3. **Cache raw responses to `data/raw/`** as soon as a call succeeds, so
   `clean.py` (task 1.4) can be developed and re-run against saved data
   without re-hitting the API every time.
4. **Run `fetch.py` from a normal home network**, not from a CI runner
   or cloud sandbox — confirmed above that this environment specifically
   cannot reach `stats.nba.com` at all.
5. No proxy/VPN workaround is being pursued for this project — it adds
   complexity and cost disproportionate to a portfolio project; the
   simpler fix (run it locally) is sufficient and easier to defend.


## Endpoints selected for V1

| Endpoint | Used by | Notes |
|---|---|---|
| `players` / `teams` (static) | id lookup for all pages | no network, always available |
| `CommonPlayerInfo` | player profile page (1.6) | bio data (position, height, draft, etc.) |
| `PlayerCareerStats` | player profile + comparison pages (1.6/1.7) | season-by-season stats |
| `LeagueGameLog` | not used in V1 | kept in mind for V2 (per-game granularity for ML features) |

Next: task 1.3 builds `fetch.py` around these endpoints and the
constraints documented above.
